In [1]:
import pandas as pd


INPUT_FILE = "incidents_clean.csv"

df = pd.read_csv(INPUT_FILE)

print(f"Loaded {len(df):,} incidents")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

required_columns = [
    "case_id",
    "variant",
    "duration_hours",
    "event_count",
    "unique_events",
    "unique_resolvers",
    "customer_satisfaction"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df["duration_hours"] = pd.to_numeric(
    df["duration_hours"],
    errors="coerce"
)

df["event_count"] = pd.to_numeric(
    df["event_count"],
    errors="coerce"
)

df["unique_events"] = pd.to_numeric(
    df["unique_events"],
    errors="coerce"
)

df["unique_resolvers"] = pd.to_numeric(
    df["unique_resolvers"],
    errors="coerce"
)

df["customer_satisfaction"] = pd.to_numeric(
    df["customer_satisfaction"],
    errors="coerce"
)

df = df.dropna(
    subset=["duration_hours", "variant"]
)


variant_summary = (
    df.groupby("variant")
    .agg(
        incidents=("case_id", "nunique"),
        mean_duration_hours=("duration_hours", "mean"),
        median_duration_hours=("duration_hours", "median"),
        p75_duration_hours=(
            "duration_hours",
            lambda x: x.quantile(0.75)
        ),
        p95_duration_hours=(
            "duration_hours",
            lambda x: x.quantile(0.95)
        ),
        mean_events=("event_count", "mean"),
        median_events=("event_count", "median"),
        mean_unique_events=("unique_events", "mean"),
        mean_unique_resolvers=("unique_resolvers", "mean"),
        mean_satisfaction=("customer_satisfaction", "mean")
    )
    .reset_index()
    .sort_values(
        "median_duration_hours",
        ascending=False
    )
)

variant_summary.to_csv(
    "variant_performance.csv",
    index=False
)


issue_summary = (
    df.groupby("issue_type")
    .agg(
        incidents=("case_id", "nunique"),
        mean_duration_hours=("duration_hours", "mean"),
        median_duration_hours=("duration_hours", "median"),
        p95_duration_hours=(
            "duration_hours",
            lambda x: x.quantile(0.95)
        ),
        mean_events=("event_count", "mean"),
        mean_resolvers=("unique_resolvers", "mean"),
        mean_satisfaction=("customer_satisfaction", "mean")
    )
    .reset_index()
    .sort_values(
        "median_duration_hours",
        ascending=False
    )
)

issue_summary.to_csv(
    "issue_type_performance.csv",
    index=False
)


channel_summary = (
    df.groupby("report_channel")
    .agg(
        incidents=("case_id", "nunique"),
        mean_duration_hours=("duration_hours", "mean"),
        median_duration_hours=("duration_hours", "median"),
        p95_duration_hours=(
            "duration_hours",
            lambda x: x.quantile(0.95)
        ),
        mean_events=("event_count", "mean"),
        mean_satisfaction=("customer_satisfaction", "mean")
    )
    .reset_index()
    .sort_values(
        "median_duration_hours",
        ascending=False
    )
)

channel_summary.to_csv(
    "channel_performance.csv",
    index=False
)


overall = df["duration_hours"]

print("\nTop variants by median resolution time:")
print(
    variant_summary[
        [
            "variant",
            "incidents",
            "mean_duration_hours",
            "median_duration_hours",
            "p75_duration_hours",
            "p95_duration_hours",
            "mean_events",
            "mean_unique_resolvers",
            "mean_satisfaction"
        ]
    ]
    .to_string(index=False)
)

print("\nIssue type performance:")
print(
    issue_summary.to_string(index=False)
)

print("\nReport channel performance:")
print(
    channel_summary.to_string(index=False)
)

print("\nOverall process:")
print(f"Incidents: {len(df):,}")
print(
    f"Average resolution time: "
    f"{overall.mean():.2f} hours"
)
print(
    f"Median resolution time: "
    f"{overall.median():.2f} hours"
)
print(
    f"75th percentile: "
    f"{overall.quantile(0.75):.2f} hours"
)
print(
    f"95th percentile: "
    f"{overall.quantile(0.95):.2f} hours"
)

print("\nSaved:")
print("- variant_performance.csv")
print("- issue_type_performance.csv")
print("- channel_performance.csv")

Loaded 31,588 incidents

Top variants by median resolution time:
   variant  incidents  mean_duration_hours  median_duration_hours  p75_duration_hours  p95_duration_hours  mean_events  mean_unique_resolvers  mean_satisfaction
 Variant 7       1261            26.642519              22.250000           41.500000           49.016667         15.0               7.534496           1.901665
 Variant 5       1188            23.328002              19.383333           35.820833           43.831667         12.0               4.863636           3.528620
Variant 13        308            22.262392              18.933333           33.541667           40.627500         11.0               4.659091           3.564935
 Variant 8        954            21.915514              18.475000           32.633333           40.273333         10.0               2.881551           3.549266
 Variant 9        625            22.241973              18.350000           33.716667           40.916667         10.0            

In [5]:
INPUT_FILE = "Incident_Management_CSV.csv"
OUTPUT_FILE = "variant_7_bottlenecks.csv"


df = pd.read_csv(
    INPUT_FILE,
    sep=";"
)

print(f"Loaded {len(df):,} event records")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

required_columns = [
    "case_id",
    "variant",
    "timestamp",
    "event"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    dayfirst=True,
    errors="coerce"
)

df = df.dropna(
    subset=["timestamp"]
)

df["variant"] = (
    df["variant"]
    .astype(str)
    .str.strip()
)

df = df[
    df["variant"] == "Variant 7"
].copy()

df = df.sort_values(
    ["case_id", "timestamp"]
).reset_index(drop=True)

incident_count = df["case_id"].nunique()

print(
    f"Analysing {incident_count:,} Variant 7 incidents"
)


df["next_event"] = (
    df.groupby("case_id")["event"]
    .shift(-1)
)

df["next_timestamp"] = (
    df.groupby("case_id")["timestamp"]
    .shift(-1)
)

df["transition_hours"] = (
    df["next_timestamp"] - df["timestamp"]
).dt.total_seconds() / 3600

transitions = df.dropna(
    subset=[
        "next_event",
        "transition_hours"
    ]
).copy()

transitions = transitions[
    transitions["transition_hours"] >= 0
].copy()

transitions["transition"] = (
    transitions["event"].astype(str)
    + " → "
    + transitions["next_event"].astype(str)
)


bottlenecks = (
    transitions
    .groupby("transition")
    .agg(
        cases=("case_id", "nunique"),
        occurrences=("transition", "size"),
        mean_hours=("transition_hours", "mean"),
        median_hours=("transition_hours", "median"),
        p75_hours=(
            "transition_hours",
            lambda x: x.quantile(0.75)
        ),
        p95_hours=(
            "transition_hours",
            lambda x: x.quantile(0.95)
        ),
        max_hours=("transition_hours", "max")
    )
    .reset_index()
)

bottlenecks["case_percentage"] = (
    bottlenecks["cases"]
    / incident_count
    * 100
)

bottlenecks = bottlenecks.sort_values(
    "median_hours",
    ascending=False
).reset_index(drop=True)

bottlenecks.insert(
    0,
    "bottleneck_rank",
    range(1, len(bottlenecks) + 1)
)

bottlenecks.to_csv(
    OUTPUT_FILE,
    index=False
)


print("\nVariant 7 bottlenecks:")

print(
    bottlenecks[
        [
            "bottleneck_rank",
            "transition",
            "cases",
            "case_percentage",
            "occurrences",
            "mean_hours",
            "median_hours",
            "p75_hours",
            "p95_hours"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

print(f"\nSaved: {OUTPUT_FILE}")

Loaded 242,901 event records
Analysing 1,261 Variant 7 incidents

Variant 7 bottlenecks:
 bottleneck_rank                                                            transition  cases  case_percentage  occurrences  mean_hours  median_hours  p75_hours  p95_hours
               1          WIP - level 1 support → Level 1 escalates to level 2 support   1261            100.0         1261    4.282474      3.883333   5.250000   8.983333
               2          WIP - level 2 support → Level 2 escalates to level 3 support   1261            100.0         2522    3.978721      3.608333   4.812500   8.616667
               3                   Ticket created → Ticket assigned to level 1 support   1261            100.0         1261    3.951282      3.500000   4.850000   8.633333
               4                            Customer feedback received → Ticket closed   1261            100.0         1261    1.061988      0.933333   1.333333   2.533333
               5            Ticket assigned to leve

In [7]:
df = pd.read_csv(
    "Incident_Management_CSV.csv",
    sep=";"
)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df["variant"] = (
    df["variant"]
    .astype(str)
    .str.strip()
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    dayfirst=True,
    errors="coerce"
)

df = df[
    df["variant"] == "Variant 7"
].copy()

df = df.sort_values(
    ["case_id", "timestamp"]
)


event_counts = (
    df.groupby(
        ["case_id", "event"]
    )
    .size()
    .reset_index(
        name="occurrences"
    )
)

summary = (
    event_counts
    .groupby("event")
    .agg(
        cases=("case_id", "nunique"),
        total_occurrences=("occurrences", "sum"),
        mean_occurrences=("occurrences", "mean"),
        max_occurrences=("occurrences", "max")
    )
    .reset_index()
    .sort_values(
        ["max_occurrences", "mean_occurrences"],
        ascending=False
    )
)

print("\nVariant 7 event repetition:")
print(
    summary.to_string(index=False)
)


print("\nExample Variant 7 event sequence:")

for case_id, group in (
    df.groupby("case_id")
    .head(30)
    .groupby("case_id")
):
    print(f"\n{case_id}")

    for _, row in group.iterrows():
        print(
            f"{row['timestamp']} | "
            f"{row['event']}"
        )

    break


Variant 7 event repetition:
                               event  cases  total_occurrences  mean_occurrences  max_occurrences
Level 2 escalates to level 3 support   1261               2522               2.0                2
  Ticket assigned to level 2 support   1261               2522               2.0                2
               WIP - level 2 support   1261               2522               2.0                2
               WIP - level 3 support   1261               2522               2.0                2
          Customer feedback received   1261               1261               1.0                1
Level 1 escalates to level 2 support   1261               1261               1.0                1
  Ticket assigned to level 1 support   1261               1261               1.0                1
                       Ticket closed   1261               1261               1.0                1
                      Ticket created   1261               1261               1.0         